# Transformer Language Model Architecture

In this notebook, we will build the next component needed to train a standard transformer language model from scratch: the transformer architecture itself.

**What we will do:** Implement a transformer language model in PyTorch.

We will build these components from scratch. In particular, we'll avoid using any definitions from `torch.nn`, `torch.nn.functional`, or `torch.optim` except for the following:
- `torch.nn.Parameter`.
- Container classes in `torch.nn` (e.g. `Module`, `ModuleList`, `Sequential`, etc.).
- The `torch.optim.Optimizer` base class.

A *language model* takes as input a batched sequence of integer token IDs, i.e. `torch.Tensor` of shape `(batch_size, sequence_length)`, and returns a (batched) normalized probability distribution over the vocabulary, i.e. a PyTorch Tensor of shape `(batch_size, sequence_length, vocab_size)`, where the predicted distribution is over the next word for each input token. When training the language model, we use these next-word predictions to calculate the cross-entropy loss between the actual next word and the predicted next word. When generating text from the language model during inference, we take the predicted next-word distribution from the final time step, i.e. the last item in the sequence, to generate the next token in the sequence (e.g. by taking the token with the highest probability, sampling from the distribution, etc.), add the generated token to the input sequence, and repeat.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from einops import rearrange, einsum

import sys; sys.path.insert(0, '..')
from tests.test_transformer import *

In [ ]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    if torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'

device = get_device()
print(f'device = {device}')

## Transformer Language Model

Given a sequence of token IDs, the transformer language model uses an input embedding to convert token IDs to dense vectors, passes the embedded tokens through num_layers transformer blocks, and then applies a learned linear projection (the "output embedding" or "LM head") to produce the predicted next-token logits. See *Figure 1* for a schematic representation.

<center>
<img src="attachment:3923ccab-ff1e-44c3-ad4c-eeec015e4bc2.png" alt="image" style="zoom:50%;"/>
</center>

### Token Embeddings
In the very first step, the transformer embeds the (batched) sequence of token IDs into a sequence of vectors containing information on the token identity (red blocks in *Figure 1*).

More specifically, given a sequence of token IDs, the transformer language model uses a token embedding layer to produce a sequence of vectors. Each embedding layer takes in a tensor of integers of shape `(batch_size, sequence_length)` and produces a sequence of vectors of shape `(batch_size, sequence_length, d_model)`.

### Pre-norm Transformer Block
After embedding, the activations are processed by several identically structured neural net layers. A standard decoder-only transformer language model consists of `num_layers` identical layers (commonly called transformer "blocks"). Each transformer block takes in an input of shape `(batch_size, sequence_length, d_model)` and returns an output of shape `(batch_size, sequence_length, d_model)`. Each block aggregates information across the sequence (via self-attention) and non-linearly transforms it (via the feed-forward layers).

### Output Normalization and Embedding
After `num_layers` transformer blocks, we will take the final activations and turn them into a distribution over the vocabulary. We will implement the "pre-norm" transformer block, which additionally requires the use of layer normalization (detailed below) after the final transformer block to ensure its outputs are properly scaled. After this normalization, we will use a standard learned linear transformation to convert the output of the transformer blocks into predicted next-token logits (see, e.g. *Radford et al. (2018) equation 2*).

## Remark: Batching, Einsum and Efficient Computation

Throughout the transformer, we will be performing the same computation applied to many batch-like inputs.
Here are a few examples:
- **Elements of a batch:** we apply the same transformer forward operation on each batch element.
- **Sequence length:** the "position-wise" operations like RMSNorm and feed-forward operate identically on each position of a sequence.
- **Attention heads:** the attention operation is batched across attention heads in a "multi-headed" attention operation.

It is useful to have an ergonomic way of performing such operations in a way that fully utilizes the GPU, and is easy to read and understand. Many PyTorch operations can take in excess “batch-like” dimensions at the start of a tensor and repeat/broadcast the operation across these dimensions efficiently.

For instance, say we are doing a position-wise, batched operation. We have a "data tensor" `D` of shape `(batch_size, sequence_length, d_model)`, and we would like to do a batched vector-matrix multiply against a matrix `A` of shape `(d_model, d_model)`. In this case, `D @ A` will do a batched matrix multiply, which is an efficient primitive in PyTorch, where the `(batch_size, sequence_length)` dimensions are
batched over.

Because of this, it is helpful to assume that your functions may be given additional batch-like dimensions and to keep those dimensions at the start of the PyTorch shape. To organize tensors so they can be batched in this manner, they might need to be shaped using many steps of view, reshape and transpose. This can be a bit of a pain, and it often gets hard to read what the code is doing and what the shapes of your tensors are.

A more ergonomic option is to use einsum notation within torch.einsum, or rather use framework agnostic libraries like einops or einx. The two key ops are einsum, which can do tensor contractions with arbitrary dimensions of input tensors, and rearrange, which can reorder, concatenate, and split arbitrary dimensions. It turns out almost all operations in machine learning are some combination of dimension juggling and tensor contraction with the occasional (usually pointwise) nonlinear function. This means that a lot of your code can be more readable and flexible when using einsum notation.

*Note:* You may benefit from learning and using einsum notation for some of these exercises. Those who have not been exposed to einsum notation before should use `einops`, while those who are already comfortable with einops should learn the more general `einx`.

Here we give some examples of how einsum notation can be used. These are a supplement to the documentation for einops, which you should read first.

**Example: Batched matrix multiplication with einops.einsum**

```python
import torch
from einops import rearrange, einsum
## Basic implementation
Y = D @ A.T
# Hard to tell the input and output shapes and what they mean.
# What shapes can D and A have, and do any of these have unexpected behavior?
## Einsum is self-documenting and robust
# D A -> Y
Y = einsum(D, A, 'batch sequence d_in, d_out d_in -> batch sequence d_out')
## Or, a batched version where D can have any leading dimensions but A is constrained.
Y = einsum(D, A, '... d_in, d_out d_in -> ... d_out')
```

**Example: Broadcasted operations with einops.rearrange**

We have a batch of images, and for each image we want to generate 10 dimmed versions based on some scaling factor:

```python
images = torch.randn(64, 128, 128, 3) # (batch, height, width, channel)
dim_by = torch.linspace(start=0.0, end=1.0, steps=10)
## Reshape and multiply
dim_value = rearrange(dim_by, 'dim_value -> 1 dim_value 1 1 1')
images_rearr = rearrange(images, 'batch height width channel -> b 1 height width channel')
dimmed_images = images_rearr * dim_value
## Or in one go:
dimmed_images = einsum(images, dim_by, 'batch height width channel, dim_value -> batch dim_value height width channel')
```

**Example: Pixel mixing with einops.rearrange**

Suppose we have a batch of images represented as a tensor of shape `(batch, height, width, channel)`, and we want to perform a linear transformation across all pixels of the image, but this transformation should happen independently for each channel. Our linear transformation is represented as a matrix `B` of shape `(height * width, height * width)`.

```python
channels_last = torch.randn(64, 32, 32, 3) # (batch, height, width, channel)
B = torch.randn(32*32, 32*32)
## Rearrange an image tensor for mixing across all pixels
channels_last_flat = channels_last.view(-1, channels_last.size(1) * channels_last.size(2), channels_last.size(3))
channels_first_flat = channels_last_flat.transpose(1, 2)
channels_first_flat_transformed = channels_first_flat @ B.T
channels_last_flat_transformed = channels_first_flat_transformed.transpose(1, 2)
channels_last_transformed = channels_last_flat_transformed.view(*channels_last.shape)
```

Instead, using einops:

```python
height = width = 32
## Rearrange replaces clunky torch view + transpose
channels_first = rearrange(channels_last, 'batch height width channel -> batch channel (height width)')
channels_first_transformed = einsum(channels_first, B, 'batch channel pixel_in, pixel_out pixel_in -> batch channel pixel_out')
channels_last_transformed = rearrange(channels_first_transformed, 'batch channel (height width) -> batch height width channel', height=height, width=width)
## Or, if you’re feeling crazy: all in one go using einx.dot (einx equivalent of einops.einsum)
height = width = 32
channels_last_transformed = einx.dot('batch row_in col_in channel, (row_out col_out) (row_in col_in) -> batch row_out col_out channel', channels_last, B, col_in=width, col_out=width)
```

The first implementation here could be improved by placing comments before and after to indicate what the input and output shapes are, but this is clunky and susceptible to bugs. With einsum notation, documentation is implementation!

Einsum notation can handle arbitrary input batching dimensions, but also has the key benefit of being self-documenting. It’s much clearer what the relevant shapes of your input and output tensors are in code that uses einsum notation. For the remaining tensors, you can consider using Tensor type hints, for instance using the `jaxtyping` library (not specific to Jax). We will talk more about the performance implications of using einsum notation in a later lesson, but for now know that they’re almost always better than the alternative!

### Mathematical Notation and Memory Ordering

Many machine learning papers use row vectors in their notation, which result in representations that mesh well with the row-major memory ordering used by default in NumPy and PyTorch. With row vectors, a linear transformation looks like

$$y = x W^T$$

for row-major $W \in \mathbb{R}^{d_{out} \times d_{in}}$ and row vector $x \in \mathbb{R}^{1 \times d_{in}}$.

In linear algebra it’s generally more common to use column vectors, where linear transformations look like

$$y = W x$$

given a row-major $W \in \mathbb{R}^{d_{out} \times d_{in}}$ and row vector $x \in \mathbb{R}^{d_{in}}$.

*We will use column vectors* for mathematical notation in this notebook, as it is generally easier to follow the math this way. You should keep in mind that if you want to use plain matrix multiplication notation, you will have to apply matrices using the row vector convention, since PyTorch uses row-major memory ordering. If you use einsum for your matrix operations, this should be a non-issue.

## Basic Building Blocks: Linear and Embedding Modules

### Parameter Initialization

Training neural networks effectively often requires careful initialization of the model parameters—bad initializations can lead to undesirable behavior such as vanishing or exploding gradients. Pre-norm transformers are unusually robust to initializations, but they can still have a siginificant impact on training speed and convergence. We will save the details for later, and instead give you some approximate initializations that should work well for most cases. For now, use:
- Linear weights: $\mathcal{N}\left(\mu=0, \sigma^2=2/(d_{in} + d_{out})\right)$ truncated at $[-3\sigma, 3\sigma]$.
- Embedding: $\mathcal{N}(\mu=0, \sigma^2=1)$ truncated at $[-3, 3]$.
- RMSNorm: $\mathbb{1}$.

You should use `torch.nn.init.trunc_normal_` to initialize the truncated normal weights.

### Linear Module

Linear layers are a fundamental building block of transformers and neural nets in general. First, you will implement your own Linear class that inherits from `torch.nn.Module` and performs a linear transformation:

$$y = W x \ .$$

Note that we do not include a bias term, following most modern LLMs.

### Problem: Implementing the linear module

Implement the `Linear` class given in the cell below. Note this class inherits from `nn.Module` and performs the linear transformation inside the `forward()` method. We will try to mimic the of the native `nn.Linear` module. As such, make sure to:
- Subclass `nn.Module` and call the superclass constructor `super().__init__()`.
- Construct and store your parameters as $W$ (not $W^T$) for memory ordering reasons, and wrap inside `nn.Parameter`.
- For parameter initialization, use `torch.nn.init.trunc_normal_` to initialize the weights.
- Avoid using `nn.Linear` or `nn.functional.linear` directly.

Once this is implemented, we will use the adapter function `run_linear` to test if your implementation works. Try to make sure all tests are passing before proceeding.

In [ ]:
class Linear(nn.Module):
    def __init__(self, in_features, out_features, device=None, dtype=None):
        """
        Construct a linear transformation module.
        Args:
            in_features: int. Final dimension of the input
            out_features: int. Final dimension of the output
            device: torch.device | None = None. Device to store the parameters on.
            dtype: torch.dtype | None = None. Data type of the parameters.
        """
        super().__init__()
        raise NotImplementedError
    
    def forward(self, x):
        """
        Apply the linear transformation to the input.
        Args:
            x: torch.Tensor
        Returns:
            torch.Tensor
        """
        raise NotImplementedError

In [ ]:
def run_linear(d_in, d_out, weights, in_features):
    """
    Given the weights of a Linear layer, compute the transformation of a batched input.
    Args:
        d_in (int): The size of the input dimension
        d_out (int): The size of the output dimension
        weights (Float[Tensor, "d_out d_in"]): The linear weights to use
        in_features (Float[Tensor, "... d_in"]): The output tensor to apply the function to
    Returns:
        Float[Tensor, "... d_out"]: The transformed output of your linear module.
    """
    layer = Linear(d_in, d_out)
    layer.weight.data = weights
    return layer(in_features)

test_linear(run_linear)

### Embedding Module

As discussed above, the first layer of the transformer is an embedding layer that maps integer token IDs into a vector space of dimension `d_model`. We will implement a custom Embedding class that inherits from `torch.nn.Module`, avoidng the use of `nn.Embedding`. The `forward()` method should select the embedding vector for each token ID by indexing into an embedding matrix of shape `(vocab_size, d_model)` using a `torch.LongTensor` of token IDs with shape `(batch_size, sequence_length)`.

### Problem: Implement the embedding module

Implement the `Embedding` class given in the cell below. Note this class inherits from `nn.Module` and performs the embedding lookup inside the `forward()` method. We will try to mimic the of the native `nn.Embedding` module. As such, make sure to:
- Subclass `nn.Module` and call the superclass constructor `super().__init__()`.
- Store the embedding matrix with the `d_model` being the final dimension
- For parameter initialization, use `torch.nn.init.trunc_normal_` to initialize the weights.
- Avoid using `nn.Embedding` or `nn.functional.embedding` directly.

Once this is implemented, we will use the adapter function `run_embedding` to test if your implementation works. Try to make sure all tests are passing before proceeding.

In [ ]:
class Embedding(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, device=None, dtype=None):
        """
        Construct an embedding module.
        Args:
            num_embeddings: int. Size of the vocabulary
            embedding_dim: int. Dimension of the embedding vectors, i.e. d_model
            device: torch.device | None = None. Device to store the parameters on. 
            dtype: torch.dtype | None = None. Data type of the parameters.
        """
        super().__init__()
        raise NotImplementedError
    
    def forward(self, token_ids):
        """
        Lookup the embedding vectors for the given token IDs.
        Args:
            token_ids: torch.Tensor
        Returns:
            torch.Tensor
        """
        raise NotImplementedError

In [ ]:
def run_embedding(vocab_size, d_model, weights, token_ids):
    """
    Given the weights of an Embedding layer, get the embeddings for a batch of token ids.
    Args:
        vocab_size (int): The number of embeddings in the vocabulary
        d_model (int): The size of the embedding dimension
        weights (Float[Tensor, "vocab_size d_model"]): The embedding vectors to fetch from
        token_ids (Int[Tensor, "..."]): The set of token ids to fetch from the Embedding layer
    Returns:
        Float[Tensor, "... d_model"]: Batch of embeddings returned by your Embedding layer.
    """
    layer = Embedding(vocab_size, d_model)
    layer.weight.data = weights
    return layer(token_ids)

test_embedding(run_embedding)

### Pre-Norm Transformer Block

Each transformer block has two sub-layers: a multi-head self-attention mechanism and a position-wise feed-forward network (*Vaswani et al., 2017, section 3.1*).

In the original transformer paper, the model uses a residual connection around each of the two sub-layers, followed by layer normalization. This architecture is commonly known as the “post-norm” transformer, since layer normalization is applied to the sublayer output. However, a variety of work has found that moving layer normalization from the output of each sub-layer to the input of each sub-layer (with an additional layer normalization after the final transformer block) improves transformer training stability (*Nguyen and
Salazar, 2019*, *Xiong et al., 2020*) - see *Figure 2* for a visual representation of this "pre-norm" transformer block. The output of each transformer block sub-layer is then added to the sub-layer input via the residual connection (*Vaswani et al., 2017, section 5.4*). An intuition for pre-norm is that there is a clean "residual stream" without any normalization going from the input embeddings to the final output of the transformer, which is purported to improve gradient flow. This pre-norm transformer is now the standard used in language
models today (e.g. GPT-3, LLaMA, PaLM, etc.), so we will implement this variant. We will walk through each of the components of a pre-norm transformer block, implementing them in sequence.

#### Root Mean Square Layer Normalization

The original transformer implementation of *Vaswani et al. (2017)* uses layer normalization (*Ba et al., 2016*) to normalize activations. Following *Touvron et al. (2023)*, we will use *root mean square layer normalization* (RMSNorm; *Zhang and Sennrich, 2019, equation 4*) for layer normalization. Given a vector $a \in \mathbb{R}^{d_{\mathrm{model}}}$ of activations, RMSNorm will rescale each activation as follows:

$$\text{RMSNorm}(a_i) = \frac{a_i}{RMS(a)} g_i \ ,$$

where

$$RMS(a) = \sqrt{\frac{1}{d_{\mathrm{model}}} \sum_{i=1}^{d_{\mathrm{model}}} a_i^2 + \varepsilon} \ .$$

Here, $g_i$ is a learnable "gain" parameter (there are `d_model` such parameters total), and $\varepsilon$ is a hyperparameter that is often fixed at `1e-5`.

You should upcast your input to torch.float32 to prevent overflow when you square the input. Overall, your forward method should look like:

```python
in_dtype = x.dtype
x = x.to(torch.float32)
# Your code here performing RMSNorm
...
result = ...
# Return the result in the original dtype
return result.to(in_dtype)
```

#### Problem: Root Mean Square Layer Normalization

Implement the `RMSNorm` class given in the cell below. Remember to upcast your input to `torch.float32` before performing the normalization (and later downcast to the original `dtype`), as described above.

Once this is implemented, we will use the adapter function `run_rmsnorm` to test if your implementation works. Try to make sure all tests are passing before proceeding.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5, device=None, dtype=None):
        """
        Construct the RMSNorm module.
        Args:
            d_model: int. Hidden dimension of the model.
            eps: float = 1e-5. Epsilon value for numerical stability.
            device: torch.device | None = None. Device to store the parameters on.
            dtype: torch.dtype | None = None. Data type of the parameters.
        """
        super().__init__()
        raise NotImplementedError
    
    def forward(self, x):
        """
        Process an input tensor of shape (batch_size, sequence_length, d_model) and return a tensor of the same shape.
        Args:
            x: torch.Tensor
        Returns:
            torch.Tensor
        """
        raise NotImplementedError

In [ ]:
def run_rmsnorm(d_model, eps, weights, in_features):
    """
    Given the weights of a RMSNorm affine transform, return the output of running RMSNorm on the input features.
    Args:
        d_model (int): The dimensionality of the RMSNorm input.
        eps: (float): A value added to the denominator for numerical stability.
        weights (Float[Tensor, "d_model"]): RMSNorm weights.
        in_features (Float[Tensor, "... d_model"]): Input features to run RMSNorm on. Can have arbitrary leading
            dimensions.
    Returns:
        Float[Tensor,"... d_model"]: Tensor of with the same shape as `in_features` with the output of running
        RMSNorm of the `in_features`.
    """
    layer = RMSNorm(d_model, eps)
    layer.weight.data = weights
    return layer(in_features)

test_rmsnorm(run_rmsnorm)

#### Position-Wise Feed-Forward Network

In the original transformer paper (*section 3.3 of Vaswani et al. 2017*), the transformer feed-forward network consists of two linear transformations with a ReLU activation $\text{ReLU}(x) = \max(0, x)$ between them. The dimensionality of the inner feed-forward layer is typically 4x the input dimensionality.

However, modern language models tend to incorporate two main changes compared to this original design: they use another activation function and employ a gating mechanism. Specifically, we will implement the "SwiGLU" activation function adopted in LLMs like Llama 3 (*Grattafiori et al., 2024*) and Qwen 2.5 (*Yang et al., 2024*), which combines the SiLU (often called Swish) activation with a gating mechanism called a
Gated Linear Unit (GLU). We will also omit the bias terms sometimes used in linear layers, following most modern LLMs since PaLM (*Chowdhery et al., 2022*) and LLaMA (*Touvron et al., 2023*).

The SiLU or Swish activation function (*Hendrycks and Gimpel, 2016, Elfwing et al., 2017*) is defined as follows:

$$\text{SiLU(x)} = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}} \ .$$

As can be seen in *Figure 3*, the SiLU activation function is similar to the ReLU activation function, but is smooth at zero.

<center>
<img src="attachment:93ee7857-d385-44e9-942b-fe76cc07b798.png" style="zoom:50%;"/>
</center>

Gated Linear Units (GLUs) were originally defined by *Dauphin et al. (2017)* as the element-wise product of a linear transformation passed through a sigmoid function and another linear transformation:

$$\text{GLU}(x, W_1, W_2) = \sigma(W_1 x) \odot W_2 x \ ,$$

where $\odot$ represents element-wise multiplication. Gated Linear Units are suggested to "reduce the vanishing gradient problem for deep architectures by providing a linear path for the gradients while retaining non-linear capabilities."

Putting the SiLU/Swish and GLU together, we get the SwiGLU, which we will use for our feed-forward networks:

$$\text{FFN}(x) = \text{SwiGLU}(x, W_1, W_2, W_3) = W_2 \left(\text{SiLU}(W_1 x) \odot W_3 x \right) \ ,$$

where $x \in \mathbb{R}^{d_{\mathrm{model}}}$; $W_1, W_2 \in \mathbb{R}^{d_{\mathrm{ff}} \times d_{\mathrm{model}}}$; $W_2 \in \mathbb{R}^{d_{\mathrm{model}} \times d_{\mathrm{ff}}}$; and canonically $d_{\mathrm{ff}} = 8/3 \cdot d_{\mathrm{model}}$.

*Shazeer (2020)* first proposed combining the SiLU/Swish activation with GLUs and conducted experiments showing that SwiGLU outperforms baselines like ReLU and SiLU (without gating) on language modeling tasks. In a later lesson, you will compare SwiGLU and SiLU. Though we’ve mentioned some heuristic arguments for these components (and the papers provide more supporting evidence), it’s good to keep an empirical perspective: a now famous quote from Shazeer’s paper is

> We offer no explanation as to why these architectures seem to work; we attribute their success,
as all else, to divine benevolence.

#### Problem: Implement the position-wise feed-forward network

Implement the `SwiGLU` class in the cell below. The SwiGLU is composed of a SiLU activation function and a GLU. In this particular case, feel free to use `torch.sigmoid` in your implementation for numerical stability purposes. Be sure to set $d_{\mathrm{ff}}$ to approximately $(8/3) d_{\mathrm{model}}$ in your implementation, while ensuring that the dimensionality of the inner feed-forward layer is a multiple of 64 to make good use of your hardware. 

Once this is implemented, we will use the adapter function `run_swiglu` to test if your implementation works. Try to make sure all tests are passing before proceeding.

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        """
        Construct the position-wise SwiGLU FFN.
        Args:
            d_model: int. Dimensionality of the feedforward input and output.
            d_ff: int. Dimensionality of the up-project happening internally to your swiglu.
        """
        super().__init__()
        raise NotImplementedError

    def forward(self, in_features):
        """
        Args:
            in_features: Float[Tensor, '... d_model']. Input embeddings to the feed-forward layer.
        Returns:
            Float[Tensor, '... d_model']. Output embeddings of the same shape as the input embeddings.
        """
        raise NotImplementedError

In [ ]:
def run_swiglu(d_model, d_ff, w1_weight, w2_weight, w3_weight, in_features):
    """
    Given the weights of a SwiGLU network, return the output of your implementation with these weights.
    Args:
        d_model: int. Dimensionality of the feedforward input and output.
        d_ff: int. Dimensionality of the up-project happening internally to your swiglu.
        w1_weight: Float[Tensor, 'd_ff d_model']. Stored weights for W1
        w2_weight: Float[Tensor, 'd_model d_ff']. Stored weights for W2
        w3_weight: Float[Tensor, 'd_ff d_model']. Stored weights for W3
        in_features: Float[Tensor, '... d_model']. Input embeddings to the feed-forward layer.
    Returns:
        Float[Tensor, '... d_model']. Output embeddings of the same shape as the input embeddings.
    """
    swiglu = SwiGLU(d_model, d_ff)
    swiglu.w1.weight.data = w1_weight
    swiglu.w2.weight.data = w2_weight
    swiglu.w3.weight.data = w3_weight
    return swiglu(in_features)

test_swiglu(run_swiglu)

#### Relative Positional Embeddings

To inject positional information into the model, we will implement Rotary Position Embeddings (*Su et al., 2021*), often called RoPE. For a given query token

$$q^{(i)} = W_q x^{(i)} \in \mathbb{R}^d$$

at token position $i$, we will apply a pairwise rotation matrix $R^i$, giving us

$$q'^{(i)} = R^i q^{(i)} = R^i W_q x^{(i)} \ .$$

Here, $R^i$ will rotate pairs of embedding elements $q_{2k-1:2k}$ as 2d vectors by the angle

$$\theta_{i,k} = \frac{i}{\Theta^{(2k-2)/d}}$$

for $k \in \{1,\ldots,d/2\}$ and some constant $\Theta$. Thus, we can consider $R^i$ to be a block-diagonal matrix of size $d \times d$, with blocks $R_k^i$ for $k \in \{1,\ldots,d/2\}$, with

$$
R_k^i =
\begin{bmatrix}
\cos(\theta_{i,k}) & -\sin(\theta_{i,k}) \\
\sin(\theta_{i,k}) & \cos(\theta_{i,k})
\end{bmatrix} \ .
$$

Thus we get the full rotation matrix

$$
R^i =
\begin{bmatrix}
R_1^i & 0 & 0 & \cdots & 0 \\
0 & R_2^i & 0 & \cdots & 0 \\
0 & 0 & R_3^i & \cdots & 0 \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
0 & 0 & 0 & \cdots & R_{d/2}^i
\end{bmatrix} \ ,
$$

where $0$'s represent $2 \times 2$ zero matrices. While one could construct the full $d \times d$ matrix, a good solution should use the properties of this matrix to implement the transformation more efficiently. Since we only care about the relative rotation of tokens within a given sequence, we can reuse the values we compute for $\cos(\theta_{i,k})$ and $\sin(\theta_{i,k})$ across layers, and different batches. If you would like to optimize it, you may use a single RoPE module referenced by all layers, and it can have a 2d pre-computed buffer of sin and cos values created during init with `self.register_buffer(persistent=False)`, instead of a `nn.Parameter` (because we do not want to learn these fixed cosine and sine values). The exact same rotation process we did for our $q^{(i)}$ is then done for $k^{(j)}$, rotating by the corresponding $R^j$. Notice that this layer has no learnable parameters.

#### Problem: Implement RoPE

Implement the `RotaryPositionalEmbedding` class below. This layer should apply the RoPE logic above to the input tensor.

Once this is implemented, we will use the adapter function `run_rope` to test if your implementation works. Try to make sure all tests are passing before proceeding.

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, theta, d_{\mathrm{K}}, max_seq_len, device=None):
        """
        Construct the RoPE module and create buffers if needed.
        Args:
            theta: float. Theta value for the RoPE
            d_{\mathrm{K}}: int. Dimension of query and key vectors.
            max_seq_len: int. Maximum sequence length that will be inputted.
            device: torch.device | None = None. Device to store the buffer on.
        """
        super().__init__()
        raise NotImplementedError
    
    def forward(self, x, token_positions):
        """
        Process an input tensor of shape (..., seq_len, d_{\mathrm{K}}) and return a tensor of the same shape.
        Note you should tolerate x with an arbitrary number of batch dimensions.
        Assume token positions are a tensor of shape (..., seq_len) specifying token positions of x along sequence dimension.
        Use the token positions to slice your (possibly precomputed) cos and sin tensors along the sequence dimension.
        Args:
            x: Float[Tensor, '... sequence_length d_{\mathrm{K}}']. Input tensor to run RoPE on.
            token_positions: Int[Tensor, '... sequence_length']. Tensor of shape (batch_size, sequence_length) with token positions.
        Returns:
            Float[Tensor, ' ... sequence_length d_{\mathrm{K}}']: Tensor with RoPE'd input.
        """
        raise NotImplementedError

In [ ]:
def run_rope(d_{\mathrm{K}}, theta, max_seq_len, x, token_positions):
    """
    Run RoPE for a given input tensor.
    Args:
        d_{\mathrm{K}} (int): Embedding dimension size for the query or key tensor.
        theta (float): RoPE parameter.
        max_seq_len (int): Maximum sequence length to pre-cache if your implementation does that.
        x (Float[Tensor, '... sequence_length d_{\mathrm{K}}']): Input tensor to run RoPE on.
        token_positions (Int[Tensor, '... sequence_length']): Tensor of shape (batch_size, sequence_length) with the token positions
    Returns:
        Float[Tensor, " ... sequence_length d_{\mathrm{K}}"]: Tensor with RoPE'd input.
    """
    rope = RotaryPositionalEmbedding(theta, d_{\mathrm{K}}, max_seq_len)
    return rope(x, token_positions)

test_rope(run_rope)

#### Scaled Dot-Product Attention

We will now implement scaled dot-product attention as described in *Vaswani et al. (2017) (section 3.2.1)*. As a preliminary step, the definition of the attention operation will make use of softmax, an operation that takes an unnormalized vector of scores and turns it into a normalized distribution,

$$\text{softmax}(v_i) = \frac{e^{v_i}}{\sum_{j=1}^n e^{v_j}} \ .$$

Note that $e^{v_i}$ can become $\infty$ for large values, and $\infty/\infty = \text{NaN}$. We can avoid this by noticing that the softmax operation is invariant to adding any constant $c$ to all inputs. We can leverage this property for numerical stability. Typically, we will subtract the largest entry of oi from all elements of $o_i$, making the new largest entry $0$. You will now implement softmax, using this trick for numerical stability.

#### Problem: Implement softmax

Fill in the `softmax` function below. This function should apply the softmax operation on a given input tensor. For numerical stability purposes, use the trick of subtracting the maximum value in the `dim` dimension from all elements of the `dim` dimension to avoid numerical stability issues.

Once this is implemented, we will use the adapter function `run_softmax` to test if your implementation works. Try to make sure all tests are passing before proceeding.

In [ ]:
def softmax(in_features, dim):
    """
    Given a tensor of inputs, return the output of softmaxing the given `dim` of the input.
    Args:
        in_features (Float[Tensor, "..."]): Input features to softmax. Shape is arbitrary.
        dim (int): Dimension of the `in_features` to apply softmax to.
    Returns:
        Float[Tensor, "..."]: Tensor of with the same shape as `in_features` with the output of
        softmax normalizing the specified `dim`.
    """
    raise NotImplementedError

In [ ]:
def run_softmax(in_features, dim):
    """
    Adaptor for the main softmax function.
    """
    return softmax(in_features, dim)

test_softmax_matches_pytorch(run_softmax)

We can now define the Attention operation mathematically as follows:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q^T K}{\sqrt{d_{\mathrm{model}}}}\right) V \ ,$$

where $Q, K, V \in \mathbb{R}^{d \times d}$. Here, $Q$, $K$, and $V$ are all inputs to this operation. Note these are not the learnable parameters. If you’re wondering why this isn’t $Q K^T$, see the comment on our mathematical convention from above.

**Masking:** It is sometimes convenient to mask the output of an attention operation. A mask should have the shape $M \in \{\text{True}, \text{False}\}^{n \times m}$, and each row $i$ of this boolean matrix indicates which keys the query $i$ should attend to. Canonically (and slightly confusingly), a value of `True` at position $(i, j)$ indicates that the query $i$ does attend to the key $j$, and a value of `False` indicates that the query does not attend to the key. In other words, "information flows" at $(i, j)$ pairs with value `True`. For example, consider a $1 \times 3$ mask matrix with entries `[[True, True, False]]`. The single query vector attends only to the first two keys.

Computationally, it will be much more efficient to use masking than to compute attention on subsequences, and we can do this by taking the pre-softmax values $Q^T K / d_{\mathrm{model}}^{1/2}$ and adding a $-\infty$ to any entry of the mask matrix that is `False`.

#### Problem: Implement scaled dot-product attention

Implement the `scaled_dot_product_attention` function below. The keys $K$ and queries $Q$ should both be the same shape `(batch_size, ..., seq_len, d_K)`, and the values $V$ should be of shape `(batch_size, ..., seq_len, d_V)`. Here, `...` represents any number of other batch-like dimensions (if provided). The function should return an output with the shape `(batch_size, ..., d_V)`. See above for a discussion on batch-like dimensions.

Your implementation should also support an optional user-provided boolean `mask` of shape `(seq_len, seq_len)`. The attention probabilities of positions with a mask value of `True` should collectively sum to one, and the attention probabilities of positions with a mask value of `False` should be zero.

Once this is implemented, we will use the adapter function `run_scaled_dot_product_attention` to test if your implementation works on 3rd and 4th order tensors. Try to make sure all tests are passing before proceeding.

In [1]:
def scaled_dot_product_attention(Q, K, V, mask):
    """
    Given key (K), query (Q), and value (V) tensors, return the output of your scaled dot product attention implementation.
    Args:
        Q (Float[Tensor, ' ... queries d_K']): Query tensor
        K (Float[Tensor, ' ... keys d_K']): Key tensor
        V (Float[Tensor, ' ... values d_V']): Values tensor
        mask (Bool[Tensor, ' ... queries keys'] | None): Mask tensor
    Returns:
        Float[Tensor, " ... queries d_V"]: Output of SDPA
    """
    raise NotImplementedError

In [ ]:
def run_scaled_dot_product_attention(Q, K, V, mask):
    """
    Adapter for main scaled_dot_product_attention function.
    """
    return scaled_dot_product_attention(Q, K, V, mask)

test_scaled_dot_product_attention(run_scaled_dot_product_attention)
test_4d_scaled_dot_product_attention(run_scaled_dot_product_attention)

#### Causal Multi-Head Self-Attention

We will implement multi-head self-attention as described in *section 3.2.2 of Vaswani et al. (2017)*. Recall that, mathematically, the operation of applying multi-head attention is defined as follows:

$$\text{MultiHead}(Q, K, V) = \text{concat}(\text{head}_1, \cdots, \text{head}_h) \ ,$$

where

$$\text{head}_i = \text{Attention}(Q_i, K_i, V_i) \ ,$$

with $Q_i$, $K_i$, $V_i$ being slice number $i \in \{1, . . . , h\}$ of size $d_{\mathrm{K}}$ or $d_{\mathrm{V}}$ of the embedding dimension for $Q$, $K$, and
$V$ respectively. With Attention being the scaled dot-product attention operation defined before. From this we can form the multi-head self-attention operation

$$\text{MultiHeadSelfAttention}(x) = W_O \text{MultiHead}(W_Q x, W_K x, W_V x) \ .$$

Here the learnable parameters are $W_Q \in \mathbb{R}^{hd_{\mathrm{K}} \times d_{\mathrm{model}}}$, $W_K \in \mathbb{R}^{hd_{\mathrm{K}} \times d_{\mathrm{model}}}$, $W_V \in \mathbb{R}^{hd_{\mathrm{V}} \times d_{\mathrm{model}}}$, and $W_O \in \mathbb{R}^{d_{\mathrm{model}} \times h d_{\mathrm{V}}}$. Since the $Q$'s, $K$'s, and $V$'s are sliced in the multi-head attention operation, we can think of $W_Q$, $W_K$, and $W_V$ as being separated for each head along the output dimension. When you have this working, you should be computing the key, value, and query projections in a total of three matrix multiplies.

*Aside:* As a stretch goal, try combining the key, query, and value projections into a single weight matrix so you only need a single
matrix multiply.

**Causal masking:** Your implementation should prevent the model from attending to future tokens in the sequence. In other words, if the model is given a token sequence $t_1, \cdots, t_n$ and we want to calculate the next-word predictions for the prefix $t_1, \cdots, t_i$ (where $i < n$), the model should not be able to access (attend to) the token representations at positions $t_{i+1}, \cdots, t_n$ since it will not have access to these tokens when generating text during inference (and these future tokens leak information about the identity of the true next word, trivializing the language modeling pre-training objective). For an input token sequence $t_1, \cdots, t_n$ we can naively prevent access to future tokens by running multi-head self-attention $n$ times (for the $n$ unique prefixes in the sequence). Instead, we’ll use causal attention masking, which allows token $i$ to attend to all positions $j \leq i$ in the sequence. You can use `torch.triu` or a broadcasted index comparison to construct this mask, and you should take advantage of the fact that your scaled dot-product attention implementation from before already supports attention masking.

**Applying RoPE:** RoPE should be applied to the query and key vectors, but not the value vectors. Also, the head dimension should be handled as a batch dimension, because in multi-head attention, attention is being applied independently for each head. This means that precisely the same RoPE rotation should be applied to the query and key vectors for each head.

#### Problem: Implement causal multi-head self-attention

Implement causal multi-head self-attention in the `MultiheadSelfAttention` class below. Folllowing *Vaswani et al. (2017)*, set $d_{\mathrm{K}} = d_{\mathrm{V}} = d_{\mathrm{model}} / h$.

Once this is implemented, we will use the adapter function `run_multihead_self_attention` to test if your implementation works. Try to make sure all tests are passing before proceeding.

In [ ]:
class MultiheadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, theta=None, max_seq_len=None):
        """
        Constructor for causal multi-head self-attention.
        Args:
            d_model: int. Dimensionality of the feedforward input and output.
            num_heads: int. Number of heads to use in multi-headed attention.
            theta: (float|None) = None. RoPE theta parameter if not None.
            max_seq_len: (int|None) = None. Maximum sequence length to pre-cache if not None.
        """
        super().__init__()
        raise NotImplementedError

    def forward(self, in_features):
        """
        Args:
            in_features: Float[Tensor, '... sequence_length d_in']. Tensor to run your implementation on.
        Returns:
            Float[Tensor, '... sequence_length d_out']. 
                Tensor with the output of running your optimized, batched multi-headed attention implementation with the given 
                QKV projection weights and input features.
        """
        raise NotImplementedError

In [ ]:
def run_multihead_self_attention(d_model, num_heads, max_seq_len, q_proj_weight, k_proj_weight, v_proj_weight, o_proj_weight, in_features):
    """
    Given the key, query, and value projection weights of a naive unbatched implementation of multi-head attention, return the output of an
    optimized batched implementation. This implementation should handle the key, query, and value projections for all heads in a single 
    matrix multiply. This function should not use RoPE. See section 3.2.2 of Vaswani et al., 2017.
    Args:
        d_model (int): Dimensionality of the feedforward input and output.
        num_heads (int): Number of heads to use in multi-headed attention.
        max_seq_len (int): Maximum sequence length to pre-cache if your implementation does that.
        q_proj_weight (Float[Tensor, 'd_K d_in']): Weights for the Q projection
        k_proj_weight (Float[Tensor, 'd_K d_in']): Weights for the K projection
        v_proj_weight (Float[Tensor, 'd_K d_in']): Weights for the V projection
        o_proj_weight (Float[Tensor, 'd_model d_V']): Weights for the output projection
        in_features (Float[Tensor, '... sequence_length d_in']): Tensor to run your implementation on.
    Returns:
        Float[Tensor, '... sequence_length d_out']: Tensor with the output of running your optimized, batched multi-headed attention
        implementation with the given QKV projection weights and input features.
    """
    attn = MultiheadSelfAttention(d_model, num_heads, theta=None, max_seq_len=max_seq_len)
    attn.q_proj.weight.data = q_proj_weight
    attn.k_proj.weight.data = k_proj_weight
    attn.v_proj.weight.data = v_proj_weight
    attn.output_proj.weight.data = o_proj_weight
    return attn(in_features)

test_multihead_self_attention(run_multihead_self_attention)

## The Full Transformer LM

Let’s begin by assembling the transformer block (it will be helpful to refer back to *Figure 2*). A transformer block contains two "sublayers", one for the multihead self attention, and another for the feed-forward network. In each sublayer, we first perform RMSNorm, then the main operation (MHA/FF), finally adding in the residual connection.

Concretely, the first half (the first "sub-layer") of the transformer block should be implementing the following set of updates to produce an output $y$ from an input $x$,

$$y = \text{MultiHeadSelfAttention}(\text{RMSNorm}(x)) \ .$$

### Problem: Implement the transformer block

Use the `TransformerBlock` class below to implement the pre-norm transformer block as described before and illustrated in *Figure 2*.

Once this is implemented, we will use the adapter function `run_transformer_block` to test if your implementation works. Try to make sure all tests are passing before proceeding.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, theta=None, max_seq_len=None):
        """
        Constructor to implement transformer block with RoPE.
        Args:
            d_model: int. Dimensionality of the transformer block inputs.
            num_heads: int. Number of heads to use in multi-head self-attention.
                Note d_model must be evenly divisible by num_heads.
            d_ff: int. Dimensionality of the position-wise feed-forward inner layer.
            theta: (float|None) = None. RoPE theta parameter if not None.
            max_seq_len: (int|None) = None. Maximum sequence length to pre-cache if not None.
        """
        super().__init__()
        raise NotImplementedError
    def forward(self, in_features):
        """
        Args:
            in_features: Float[Tensor, 'batch sequence_length d_model']. Tensor of input features.
        Returns:
            Float[Tensor, 'batch sequence_length d_model']. Tensor with the output features.
        """
        raise NotImplementedError

In [ ]:
# Note: This requires the TransformerBlock starter code to use these attribute names:
# - self.attn = MultiheadSelfAttention(d_model, num_heads, theta, max_seq_len)
# - self.ln1 = RMSNorm(d_model)
# - self.ffn = SwiGLU(d_model, d_ff)
# - self.ln2 = RMSNorm(d_model)
# And MultiheadSelfAttention must use q_proj, k_proj, v_proj, output_proj as attribute names, and SwiGLU must use w1, w2, w3.

def run_transformer_block(d_model, num_heads, d_ff, max_seq_len, theta, weights, in_features):
    """
    Given the weights of a pre-norm transformer block and input features, return the output of running the transformer block on the input features.
    This function should use RoPE. Depending on your implementation, you may simply need to pass the relevant args to your TransformerBlock
    constructor, or you may need to initialize your own RoPE class and pass that instead.
    Args:
        d_model (int): The dimensionality of the transformer block input.
        num_heads (int): Number of heads to use in multi-headed attention. `d_model` must be
            evenly divisible by `num_heads`.
        d_ff (int): Dimensionality of the feed-forward inner layer.
        max_seq_len (int): Maximum sequence length to pre-cache if your implementation does that.
        theta (float): RoPE parameter.
        weights (dict[str, Tensor]):
            State dict of our reference implementation.
            The keys of this dictionary are:
            - `attn.q_proj.weight`
                The query projections for all `num_heads` attention heads.
                Shape is (d_model, d_model).
                The rows are ordered by matrices of shape (num_heads, d_{\mathrm{K}}),
                so `attn.q_proj.weight == torch.cat([q_heads.0.weight, ..., q_heads.N.weight], dim=0)`.
            - `attn.k_proj.weight`
                The key projections for all `num_heads` attention heads.
                Shape is (d_model, d_model).
                The rows are ordered by matrices of shape (num_heads, d_{\mathrm{K}}),
                so `attn.k_proj.weight == torch.cat([k_heads.0.weight, ..., k_heads.N.weight], dim=0)`.
            - `attn.v_proj.weight`
                The value projections for all `num_heads` attention heads.
                Shape is (d_model, d_model).
                The rows are ordered by matrices of shape (num_heads, d_{\mathrm{V}}),
                so `attn.v_proj.weight == torch.cat([v_heads.0.weight, ..., v_heads.N.weight], dim=0)`.
            - `attn.output_proj.weight`
                Weight of the multi-head self-attention output projection
                Shape is (d_model, d_model).
            - `ln1.weight`
                Weights of affine transform for the first RMSNorm
                applied in the transformer block.
                Shape is (d_model,).
            - `ffn.w1.weight`
                Weight of the first linear transformation in the FFN.
                Shape is (d_model, d_ff).
            - `ffn.w2.weight`
                Weight of the second linear transformation in the FFN.
                Shape is (d_ff, d_model).
            - `ffn.w3.weight`
                Weight of the third linear transformation in the FFN.
                Shape is (d_model, d_ff).
            - `ln2.weight`
                Weights of affine transform for the second RMSNorm
                applied in the transformer block.
                Shape is (d_model,).
        in_features (Float[Tensor, "batch sequence_length d_model"]):
            Tensor to run your implementation on.
    Returns:
        Float[Tensor, "batch sequence_length d_model"] Tensor with the output of
        running the Transformer block on the input features while using RoPE.
    """
    block = TransformerBlock(d_model, num_heads, d_ff, theta, max_seq_len=max_seq_len)
    block.load_state_dict(weights)
    return block(in_features)

test_transformer_block(run_transformer_block)

### Problem: Implementing the transformer LM

Now it's time to put everything together to get the full LM. Use the `TransformerLM` class below to implement the full LM using the high-level diagram in *Figure 1*. Following our description of the embedding before, feed this into `num_layers` transformer blocks, and then pass that into the three output layers to obtain a distribution over the vocabulary.

Once this is implemented, we will use the adapter function `run_transformer_lm` to test if your implementation works. Try to make sure all tests are passing before proceeding.

In [ ]:
class TransformerLM(nn.Module):
    def __init__(self, vocab_size, context_length, num_layers, d_model, num_heads, d_ff, theta=None, max_seq_len=None):
        """
        Constructor to implement a transformer LM.
        Args:
            vocab_size: int. Vocabulary size, necessary to determine the dimensionality of the token embedding matrix.
            context_length: int. Maximum context length, necessary to determine the dimensionality of the position embedding matrix.
            num_layers: int. Number of transformer blocks to use.
            d_model: int. Dimensionality of the transformer block inputs.
            num_heads: int. Number of heads to use in multi-head self-attention.
                Note d_model must be evenly divisible by num_heads.
            d_ff: int. Dimensionality of the position-wise feed-forward inner layer.
            theta: (float|None) = None. RoPE theta parameter if not None.
            max_seq_len: (int|None) = None. Maximum sequence length to pre-cache if not None.
        """
        super().__init__()
        raise NotImplementedError
    def forward(self, in_indices):
        """
        Args:
            in_indices: Int[Tensor, 'batch_size sequence_length']. Tensor with input indices to run the LM. 
                Shape is (batch_size, sequence_length), where sequence_length <= context_length.
        Returns:
            Float[Tensor, 'batch_size sequence_length vocab_size']. 
                Tensor with the predicted unnormalized next-word distribution for each token.
        """
        raise NotImplementedError

In [ ]:
# Note: This requires TransformerLM to use:
# - self.token_embeddings = Embedding(vocab_size, d_model)
# - self.layers = nn.ModuleList([TransformerBlock(...) for _ in range(num_layers)])
# - self.ln_final = RMSNorm(d_model)
# - self.lm_head = Linear(d_model, vocab_size)

def run_transformer_lm(vocab_size, context_length, d_model, num_layers, num_heads, d_ff, rope_theta, weights, in_indices):
    """
    Given the weights of a transformer language model and input indices, return the output of running a forward pass on the input indices.
    This function should use RoPE.
    Args:
        vocab_size (int): The number of unique items in the output vocabulary to be predicted.
        context_length (int): The maximum number of tokens to process at once.
        d_model (int): The dimensionality of the model embeddings and sublayer outputs.
        num_layers (int): The number of transformer layers to use.
        num_heads (int): Number of heads to use in multi-headed attention. `d_model` must be
            evenly divisible by `num_heads`.
        d_ff (int): Dimensionality of the feed-forward inner layer (section 3.3).
        rope_theta (float): The RoPE $\Theta$ parameter.
        weights (dict[str, Tensor]):
            State dict of our reference implementation. {num_layers} refers to an
            integer between `0` and `num_layers - 1` (the layer index).
            The keys of this dictionary are:
            - `token_embeddings.weight`
                Token embedding matrix. Shape is (vocab_size, d_model).
            - `layers.{num_layers}.attn.q_proj.weight`
                The query projections for all `num_heads` attention heads.
                Shape is (num_heads * (d_model / num_heads), d_model).
                The rows are ordered by matrices of shape (num_heads, d_{\mathrm{K}}),
                so `attn.q_proj.weight == torch.cat([q_heads.0.weight, ..., q_heads.N.weight], dim=0)`.
            - `layers.{num_layers}.attn.k_proj.weight`
                The key projections for all `num_heads` attention heads.
                Shape is (num_heads * (d_model / num_heads), d_model).
                The rows are ordered by matrices of shape (num_heads, d_{\mathrm{K}}),
                so `attn.k_proj.weight == torch.cat([k_heads.0.weight, ..., k_heads.N.weight], dim=0)`.
            - `layers.{num_layers}.attn.v_proj.weight`
                The value projections for all `num_heads` attention heads.
                Shape is (num_heads * (d_model / num_heads), d_model).
                The rows are ordered by matrices of shape (num_heads, d_{\mathrm{V}}),
                so `attn.v_proj.weight == torch.cat([v_heads.0.weight, ..., v_heads.N.weight], dim=0)`.
            - `layers.{num_layers}.attn.output_proj.weight`
                Weight of the multi-head self-attention output projection
                Shape is ((d_model / num_heads) * num_heads, d_model).
            - `layers.{num_layers}.ln1.weight`
                Weights of affine transform for the first RMSNorm
                applied in the transformer block.
                Shape is (d_model,).
            - `layers.{num_layers}.ffn.w1.weight`
                Weight of the first linear transformation in the FFN.
                Shape is (d_model, d_ff).
            - `layers.{num_layers}.ffn.w2.weight`
                Weight of the second linear transformation in the FFN.
                Shape is (d_ff, d_model).
            - `layers.{num_layers}.ffn.w3.weight`
                Weight of the third linear transformation in the FFN.
                Shape is (d_model, d_ff).
            - `layers.{num_layers}.ln2.weight`
                Weights of affine transform for the second RMSNorm
                applied in the transformer block.
                Shape is (d_model,).
            - `ln_final.weight`
                Weights of affine transform for RMSNorm applied to the output of the final transformer block.
                Shape is (d_model, ).
            - `lm_head.weight`
                Weights of the language model output embedding.
                Shape is (vocab_size, d_model).
        in_indices (Int[Tensor, "batch_size sequence_length"]) Tensor with input indices to run the language model on. Shape is (batch_size, sequence_length), where
            `sequence_length` is at most `context_length`.
    Returns:
        Float[Tensor, "batch_size sequence_length vocab_size"]: Tensor with the predicted unnormalized
        next-word distribution for each token.
    """
    lm = TransformerLM(vocab_size, context_length, num_layers, d_model, num_heads, d_ff, rope_theta)
    lm.load_state_dict(weights)
    return lm(in_indices)

test_transformer_lm(run_transformer_lm)
test_transformer_lm_truncated_input(run_transformer_lm)

### Problem: Transformer LM resource accounting

**Resource accounting:** It is useful to be able to understand how the various parts of the transformer consume compute and memory. We will go through the steps to do some basic "FLOPs accounting." The *vast* majority of FLOPS in a transformer are matrix multiplies, so our core approach is simple:
1. Write down all the matrix multiplies in a transformer forward pass.
2. Convert each matrix multiply into FLOPs required.

For this second step, the following fact will be useful:

**Rule:** Given $A \in \mathbb{R}^{m \times n}$ and $B \in \mathbb{R}^{n \times p}$, the matrix-matrix product $AB$ requires $2mnp$ FLOPs.

To see this, note that $(AB)[i, j] = A[i, :] · B[:, j]$, and that this dot product requires $n$ additions and $n$ multiplications ($2n$ FLOPs). Then, since the matrix-matrix product AB has m×p entries, the total number of FLOPS is $(2n)(mp) = 2mnp$.

Before you do the following problem, it can be helpful to go through each component of your transformer block and transformer LM, and list out all the matrix multiplies and their associated FLOPs costs.

1. Consider GPT-2 XL, which has the configurations below. Suppose we constructed our model using this configuration. How many trainable parameters would our model have? Assuming each parameter is represented using single-precision floating point, how much memory is required to just load this model?
    - `vocab_size`: 50,257
    - `context_length`: 1,024
    - `num_layers`: 48
    - `d_model`: 1,600
    - `num_heads`: 25
    - `d_ff`: 6,400
2. Identify the matrix multiplies required to complete a forward pass of our GPT-2 XL-shaped model. How many FLOPs do these matrix multiplies require in total? Assume that our input sequence has `context_length` tokens. Provide a list of matrix multiplies (with descriptions), and the total number of FLOPs required.
3. Based on your analysis above, which parts of the model require the most FLOPs?
4. Repeat your analysis with GPT-2 small (12 layers, 768 `d_model`, 12 heads), GPT-2 medium (24 layers, 1024 `d_model`, 16 heads), and GPT-2 large (36 layers, 1280 `d_model`, 20 heads). As the model size increases, which parts of the transformer LM take up proportionally more or less of the total FLOPs? For each model, provide a breakdown of model components and its associated FLOPs (as a proportion of the total FLOPs required for a forward pass). In addition, provide a one-to-two sentence description of how varying the model size changes the proportional FLOPs of each component.
5. Take GPT-2 XL and increase the context length to 16,384. How does the total FLOPs for one forward pass change? How do the relative contribution of FLOPs of the model components change?

In [ ]:
# answers

**Temporary Note:**

```
Class: Linear
Attributes: self.weight
────────────────────────────────────────
Class: Embedding
Attributes: self.weight
────────────────────────────────────────
Class: RMSNorm
Attributes: self.weight
────────────────────────────────────────
Class: SwiGLU
Attributes: self.w1, self.w2, self.w3 (each a Linear)
────────────────────────────────────────
Class: RotaryPositionalEmbedding
Attributes: (no learnable params, buffers only)
────────────────────────────────────────
Class: MultiheadSelfAttention
Attributes: self.q_proj, self.k_proj, self.v_proj, self.output_proj (each a Linear)
────────────────────────────────────────
Class: TransformerBlock
Attributes: self.attn (MHA), self.ln1, self.ln2 (RMSNorm), self.ffn (SwiGLU)
────────────────────────────────────────
Class: TransformerLM
Attributes: self.token_embeddings (Embedding), self.layers (ModuleList of TransformerBlock), self.ln_final (RMSNorm), self.lm_head (Linear)
```

```
Docstring shape errors in cell 32 and 35                                                                                                               
The weight shape docs in the run_transformer_block and run_transformer_lm adapters have the FFN weight shapes swapped:

  ┌───────────────┬─────────────────┬─────────────────┐
  │      Key      │ Currently says  │    Should be    │
  ├───────────────┼─────────────────┼─────────────────┤
  │ ffn.w1.weight │ (d_model, d_ff) │ (d_ff, d_model) │
  ├───────────────┼─────────────────┼─────────────────┤
  │ ffn.w2.weight │ (d_ff, d_model) │ (d_model, d_ff) │
  ├───────────────┼─────────────────┼─────────────────┤
  │ ffn.w3.weight │ (d_model, d_ff) │ (d_ff, d_model) │
  └───────────────┴─────────────────┴─────────────────┘

This matches the SwiGLU math: 
- W1 and W3 project from d_model up to d_ff, so they're (d_ff, d_model). 
- W2 projects back down, so it's (d_model, d_ff).
The run_swiglu adapter (cell 17) has the correct shapes.
```